In [ ]:
#
#! TEMP - i

# Load Data
# combine X_train and X_test (add _[test/train])

N_train = 100
path_trainfold = os.path.join(project_root, "data/processed/bootstrapped_sameSplit/N_{}".format(N_train)) 
N_test = 1000
path_testfold = os.path.join(project_root, "data/processed/bootstrapped_sameSplit/N_{}".format(N_test)) 

train_1 = pd.read_pickle(path_trainfold + "/train_1.pkl")
train_2 = pd.read_pickle(path_trainfold + "/train_2.pkl")
train_3 = pd.read_pickle(path_trainfold + "/train_3.pkl")
test = pd.read_pickle(path_testfold + "/test.pkl")
# test = pd.read_pickle(test_dir + "/test.pkl")
features = train_1.columns[:96]

train_1.index = 'train1_'+ train_1['Gene_KO'] + train_1['boot'].replace({False: '_real', True: '_boot'})
train_2.index = 'train2_'+ train_2['Gene_KO'] + train_2['boot'].replace({False: '_real', True: '_boot'})
train_3.index = 'train3_'+ train_3['Gene_KO'] + train_3['boot'].replace({False: '_real', True: '_boot'})
test.index = 'test_'+ test['Gene_KO'] + test['boot'].replace({False: '_real', True: '_boot'})

X_train1 = train_1.loc[:,features]
y_train1 = train_1.loc[:, "label"]
X_train2 = train_2.loc[:,features]
y_train2 = train_2.loc[:, "label"]
X_train3 = train_3.loc[:,features]
y_train3 = train_3.loc[:, "label"]
X_test = test.loc[:,features]
y_test = test.loc[:, "label"]

# Concatenate the DataFrames along the columns (axis=1)
df_X_all = pd.concat([X_train1.T, X_train2.T, X_train3.T, X_test.T], axis=1)

df_X_all = df_X_all * 1000
df_X_all.index.name = 'Mutation Types'

path_X_all = os.path.join(project_root,"data/processed/bootstrapped_sameSplit_sorted/X_allx1000_new.text")
df_X_all.to_csv(path_X_all, sep="\t")



In [ ]:
import os
import pandas as pd

# --- Configuration ---
project_root = "/Users/sande/Documents/GitHub/SNMF"
N_train = 100
N_test = 1000
path_trainfold = os.path.join(project_root, f"data/processed/bootstrapped_sameSplit/N_{N_train}")
path_testfold = os.path.join(project_root, f"data/processed/bootstrapped_sameSplit/N_{N_test}")
output_real_dir = os.path.join(project_root, "data/processed/real")
os.makedirs(output_real_dir, exist_ok=True)

# --- Load data ---
train_1 = pd.read_pickle(os.path.join(path_trainfold, "train_1.pkl"))
train_2 = pd.read_pickle(os.path.join(path_trainfold, "train_2.pkl"))
train_3 = pd.read_pickle(os.path.join(path_trainfold, "train_3.pkl"))
test = pd.read_pickle(os.path.join(path_testfold, "test.pkl"))

# --- Add source label ---
train_1["source"] = "train1"
train_2["source"] = "train2"
train_3["source"] = "train3"
test["source"] = "test"

# --- Combine all data ---
df_all = pd.concat([train_1, train_2, train_3, test], axis=0)

# --- Filter to real samples only ---
df_real = df_all[df_all["boot"] == False].copy()

# --- Add split column (train/test) ---
df_real["split"] = df_real["source"].apply(lambda s: "train" if "train" in s else "test")

# --- Create Gene_KO_paired column with unique identifiers ---
df_real["Gene_KO_paired"] = (
    df_real.groupby("Gene_KO").cumcount().astype(str)
)
df_real["Gene_KO_paired"] = df_real["Gene_KO"] + "_" + df_real["Gene_KO_paired"]

# --- Set index for clarity ---
df_real.index = df_real["source"] + "_" + df_real["Gene_KO_paired"] + "_real"

# --- Save full DataFrame ---
output_path = os.path.join(output_real_dir, "zou_all.pkl")
df_real.to_pickle(output_path)

print(f"✅ Saved full real data with unique KO identifiers to: {output_path}")
